# 21/ Movie Recommendation System

**Dataset (Kaggle):** [https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset)

> Use ratings_small.csv (100k ratings, 700 users, 9000 movies) from The Movies Dataset; movies_metadata.csv has titles/genres if you want to map movieId to a title.

**Goal:** Recommend movies to a user with collaborative filtering based on similar users' ratings.

**How to get the data:**
1. Download from the link above (or run `kaggle datasets download -d <dataset-slug> -p data/ --unzip`, requires a `kaggle.json` API token in `~/.kaggle/`).
2. Place the extracted file(s) in a local `data/` folder next to this notebook.
3. Update `DATA_PATH` in the cell below if your filename differs.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")


In [ ]:
ratings = pd.read_csv("data/ratings_small.csv")   # userId, movieId, rating, timestamp
print(ratings.shape)
ratings.head()


## Exploratory Data Analysis

In [ ]:
print("Users:", ratings['userId'].nunique(), " Items:", ratings['movieId'].nunique(), " Ratings:", len(ratings))
ratings['rating'].hist(bins=10)
plt.title("Rating distribution")
plt.show()


## Build User-Item Matrix

In [ ]:
user_item = ratings.pivot_table(index='userId', columns='movieId', values='rating')
print(user_item.shape)

# Keep only items/users with enough activity for a manageable demo matrix
min_ratings_per_item = 20
popular_items = user_item.columns[user_item.count() >= min_ratings_per_item]
user_item = user_item[popular_items]
user_item_filled = user_item.fillna(0)


## Item-Based Collaborative Filtering (Cosine Similarity)

In [ ]:
item_similarity = cosine_similarity(user_item_filled.T)
item_sim_df = pd.DataFrame(item_similarity, index=user_item.columns, columns=user_item.columns)

def recommend_similar_items(item_id, top_n=10):
    if item_id not in item_sim_df.columns:
        return f"Item {item_id} not found in the filtered matrix."
    return item_sim_df[item_id].sort_values(ascending=False)[1:top_n+1]

# Example: recommend items similar to the first popular item
sample_item = user_item.columns[0]
recommend_similar_items(sample_item)


## Recommend for a Specific User

In [ ]:
def recommend_for_user(user_id, top_n=10):
    if user_id not in user_item.index:
        return f"User {user_id} not found."
    user_ratings = user_item.loc[user_id].dropna()
    scores = pd.Series(dtype=float)
    for item, rating in user_ratings.items():
        sims = item_sim_df[item] * rating
        scores = scores.add(sims, fill_value=0)
    scores = scores.drop(user_ratings.index, errors='ignore')
    return scores.sort_values(ascending=False).head(top_n)

sample_user = user_item.index[0]
recommend_for_user(sample_user)


## Next Steps
- Try matrix factorization (`surprise` library's `SVD`) for better scalability.
- Evaluate with RMSE / precision@k on a held-out split.
- Combine with content-based features (genres, metadata) for a hybrid recommender.